In [1]:
import numpy as np
from ann_engine.layers.layers import Dense
from ann_engine.layers import Sequential
from ann_engine.optimizers import Adam

In [2]:
X_train = np.random.randn(1000, 784)
y_train = np.random.randint(0, 10, (1000, 10)) 

In [3]:
model1 = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

In [4]:
print(model1)

In [5]:
model1.compile(optimizer=Adam(lr=0.01),
             loss = 'categorical_crossentropy',
             metrics = ['accuracy'])

In [6]:
history_of_ann_engine = model1.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)

Epoch 1/100  loss: 108.5442  val_loss: 101.0192  accuracy: 0.1125  val_accuracy: 0.1250
Epoch 11/100  loss: 104.6652  val_loss: 101.0528  accuracy: 0.1425  val_accuracy: 0.1150
Epoch 21/100  loss: 104.5809  val_loss: 101.0369  accuracy: 0.1175  val_accuracy: 0.1400
Epoch 31/100  loss: 104.6371  val_loss: 101.0417  accuracy: 0.1237  val_accuracy: 0.1250
Epoch 41/100  loss: 104.7033  val_loss: 101.0968  accuracy: 0.1163  val_accuracy: 0.1350
Epoch 51/100  loss: 104.7459  val_loss: 101.1260  accuracy: 0.1175  val_accuracy: 0.1300
Epoch 61/100  loss: 104.7707  val_loss: 101.1734  accuracy: 0.1300  val_accuracy: 0.1400
Epoch 71/100  loss: 105.1690  val_loss: 101.3446  accuracy: 0.0975  val_accuracy: 0.1350
Epoch 81/100  loss: 104.9682  val_loss: 101.1350  accuracy: 0.1325  val_accuracy: 0.1600
Epoch 91/100  loss: 104.8793  val_loss: 101.0096  accuracy: 0.1462  val_accuracy: 0.1700
Epoch 100/100  loss: 104.8232  val_loss: 101.0396  accuracy: 0.1363  val_accuracy: 0.1650


In [7]:
model1.summary()


Model: Sequential
Layer (type)                        Output Shape              Param #   
----------------------------------------------------------------------
0. Dense                            (1, 128)                  100,480   
1. Dense                            (1, 64)                   8,256     
2. Dense                            (1, 10)                   650       
Total params:         109,386
Trainable params:     109,386
Non-trainable params: 0


In [8]:
"""
test_training.py
================
End-to-end training tests for ann_engine.

Tests:
    1. Multi-class classification  (random data, 10 classes)
    2. Binary classification       (XOR-style problem)
    3. Regression                  (sine wave)
    4. Gradient sanity check       (norms must be non-zero)
    5. Predict & Evaluate          (API smoke test)
"""

import sys
import numpy as np

# ── adjust this path if your project root is elsewhere ───────────────────────
sys.path.insert(0, '.')

from ann_engine.layers.layers import Dense
from ann_engine.layers.sequential import Sequential
from ann_engine.optimizers.adam import Adam

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
RESET  = "\033[0m"
BOLD   = "\033[1m"

passed = 0
failed = 0

def section(title):
    print(f"\n{'='*60}")
    print(f"  {BOLD}{title}{RESET}")
    print(f"{'='*60}")

def check(name, condition, detail=""):
    global passed, failed
    if condition:
        print(f"  {GREEN}✓ PASS{RESET}  {name}")
        passed += 1
    else:
        print(f"  {RED}✗ FAIL{RESET}  {name}  {YELLOW}{detail}{RESET}")
        failed += 1

def make_model(layers, loss, lr=0.01, metrics=None):
    model = Sequential(layers)
    model.compile(optimizer=Adam(lr=lr), loss=loss, metrics=metrics or [])
    return model

# ─────────────────────────────────────────────────────────────────────────────
# Test 1 — Multi-class classification (10 classes, random data)
# ─────────────────────────────────────────────────────────────────────────────
section("Test 1: Multi-class Classification (10 classes)")

np.random.seed(0)
X_cls = np.random.randn(800, 64).astype(np.float32)
y_idx = np.random.randint(0, 10, 800)
y_cls = np.eye(10)[y_idx].astype(np.float32)

model_cls = make_model([
    Dense(64, activation='relu', input_shape=(64,)),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax'),
], loss='categorical_crossentropy', lr=0.01, metrics=['accuracy'])

h = model_cls.fit(X_cls, y_cls, epochs=40, batch_size=64,
                   validation_split=0.2, verbose=0)

loss_dropped   = h['loss'][-1] < h['loss'][0]
acc_increased  = h['metric_0'][-1] > h['metric_0'][0]
loss_gt_zero   = h['loss'][-1] > 0
final_acc      = h['metric_0'][-1]
first_loss     = h['loss'][0]
final_loss     = h['loss'][-1]

print(f"  Loss  : {first_loss:.4f} → {final_loss:.4f}")
print(f"  Acc   : {h['metric_0'][0]:.4f} → {final_acc:.4f}")

check("Loss decreased over training",      loss_dropped,
      f"first={first_loss:.4f} last={final_loss:.4f}")
check("Accuracy increased over training",  acc_increased)
check("Loss is positive (not NaN/zero)",   loss_gt_zero and not np.isnan(final_loss))
check("History has 'loss' key",            'loss' in h)
check("History has 'val_loss' key",        'val_loss' in h)
check("History length matches epochs",     len(h['loss']) == 40)

# ─────────────────────────────────────────────────────────────────────────────
# Test 2 — Binary classification (linearly separable)
# ─────────────────────────────────────────────────────────────────────────────
section("Test 2: Binary Classification (linearly separable)")

np.random.seed(1)
X_bin = np.random.randn(600, 8).astype(np.float32)
# label = 1 if sum of features > 0 else 0  (easy linear boundary)
y_bin = (X_bin.sum(axis=1) > 0).astype(np.float32).reshape(-1, 1)

model_bin = make_model([
    Dense(16, activation='relu', input_shape=(8,)),
    Dense(8,  activation='relu'),
    Dense(1,  activation='sigmoid'),
], loss='bce', lr=0.01, metrics=['accuracy'])

h_bin = model_bin.fit(X_bin, y_bin, epochs=50, batch_size=32,
                       validation_split=0.2, verbose=0)

first_loss_b = h_bin['loss'][0]
final_loss_b = h_bin['loss'][-1]
final_acc_b  = h_bin['metric_0'][-1]

print(f"  Loss  : {first_loss_b:.4f} → {final_loss_b:.4f}")
print(f"  Acc   : {h_bin['metric_0'][0]:.4f} → {final_acc_b:.4f}")

check("Loss decreased",             final_loss_b < first_loss_b)
check("Final accuracy > 80%",       final_acc_b > 0.80,
      f"got {final_acc_b:.4f}")
check("No NaN in loss history",     not any(np.isnan(v) for v in h_bin['loss']))

# ─────────────────────────────────────────────────────────────────────────────
# Test 3 — Regression (sine wave)
# ─────────────────────────────────────────────────────────────────────────────
section("Test 3: Regression (sine wave)")

np.random.seed(2)
X_reg = np.linspace(-np.pi, np.pi, 500).reshape(-1, 1).astype(np.float32)
y_reg = np.sin(X_reg).astype(np.float32)

model_reg = make_model([
    Dense(32, activation='relu', input_shape=(1,)),
    Dense(32, activation='tanh'),
    Dense(1,  activation=None),
], loss='mse', lr=0.005, metrics=['mae'])

h_reg = model_reg.fit(X_reg, y_reg, epochs=100, batch_size=32,
                       validation_split=0.1, verbose=0)

first_mse = h_reg['loss'][0]
final_mse = h_reg['loss'][-1]
final_mae = h_reg['metric_0'][-1]

print(f"  MSE   : {first_mse:.4f} → {final_mse:.4f}")
print(f"  MAE   : {h_reg['metric_0'][0]:.4f} → {final_mae:.4f}")

check("MSE decreased",              final_mse < first_mse)
check("Final MAE < 0.15",           final_mae < 0.15,
      f"got {final_mae:.4f}")
check("No NaN in loss history",     not any(np.isnan(v) for v in h_reg['loss']))

# ─────────────────────────────────────────────────────────────────────────────
# Test 4 — Gradient sanity check
# ─────────────────────────────────────────────────────────────────────────────
section("Test 4: Gradient Sanity Check")

# Force-reload losses module to bypass any stale .pyc cache
import importlib
import ann_engine.losses.loss as _ll_mod
importlib.reload(_ll_mod)
# Re-patch the loss on the model after reload
from ann_engine.core import Tensor, Parameter

np.random.seed(3)
X_g = np.random.randn(32, 16).astype(np.float32)
y_g = np.eye(4)[np.random.randint(0, 4, 32)].astype(np.float32)

model_g = make_model([
    Dense(8,  activation='relu', input_shape=(16,)),
    Dense(4,  activation='softmax'),
], loss='categorical_crossentropy', lr=0.01)

model_g.build((32, 16))

xb = Tensor(X_g)
yb = Tensor(y_g)

# zero_grad BEFORE forward — not after backward
model_g._optimizer.zero_grad()
pred = model_g.forward(xb)
loss = model_g._loss(pred, yb)
loss.backward()

# Verify computation graph actually reaches Parameters
_topo, _visited = [], set()
def _build_topo(t):
    if id(t) not in _visited:
        _visited.add(id(t))
        for child in t._prev:
            _build_topo(child)
        _topo.append(t)
_build_topo(loss)
_param_nodes = [t for t in _topo if isinstance(t, Parameter)]

print(f"  Loss value      : {float(loss.data):.4f}")
print(f"  Graph nodes     : {len(_topo)}")
print(f"  Parameter nodes : {len(_param_nodes)}")
check("Computation graph reaches model Parameters",
      len(_param_nodes) > 0,
      f"found {len(_param_nodes)} — loss is disconnected (broken graph in loss fn)")

for i, layer in enumerate(model_g.layers):
    if hasattr(layer, 'W') and layer.W is not None:
        w_norm = np.linalg.norm(layer.W.grad)
        b_norm = np.linalg.norm(layer.b.grad) if layer.b is not None else 0.0
        print(f"  Layer {i}  W grad norm: {w_norm:.6f}  b grad norm: {b_norm:.6f}")
        check(f"Layer {i} W has non-zero gradient", w_norm > 0,
              f"norm={w_norm:.6f}")
        check(f"Layer {i} b has non-zero gradient", b_norm > 0,
              f"norm={b_norm:.6f}")

check("Loss is finite", np.isfinite(float(loss.data)))

# ─────────────────────────────────────────────────────────────────────────────
# Test 5 — predict() and evaluate() API
# ─────────────────────────────────────────────────────────────────────────────
section("Test 5: predict() and evaluate() API")

np.random.seed(4)
X_t = np.random.randn(100, 64).astype(np.float32)
y_t = np.eye(10)[np.random.randint(0, 10, 100)].astype(np.float32)

# Use already-trained cls model
preds = model_cls.predict(X_t)
results = model_cls.evaluate(X_t, y_t, batch_size=32)

check("predict() returns Tensor",             hasattr(preds, 'data'))
check("predict() output shape is (100, 10)",  preds.data.shape == (100, 10),
      f"got {preds.data.shape}")
check("Softmax output sums to ~1 per row",
      np.allclose(preds.data.sum(axis=1), 1.0, atol=1e-4))
check("evaluate() returns list",              isinstance(results, list))
check("evaluate() returns [loss, metric, …]", len(results) >= 1)
check("evaluate() loss is finite",            np.isfinite(results[0]))

# ─────────────────────────────────────────────────────────────────────────────
# Test 6 — model.summary() doesn't crash
# ─────────────────────────────────────────────────────────────────────────────
section("Test 6: summary() smoke test")

try:
    model_cls.summary()
    check("summary() runs without error", True)
except Exception as e:
    check("summary() runs without error", False, str(e))

# ─────────────────────────────────────────────────────────────────────────────
# Final report
# ─────────────────────────────────────────────────────────────────────────────
total = passed + failed
print(f"\n{'='*60}")
print(f"  {BOLD}Results: {GREEN}{passed}{RESET}{BOLD} passed / "
      f"{RED}{failed}{RESET}{BOLD} failed  ({total} total){RESET}")
print(f"{'='*60}\n")

if failed > 0:
    sys.exit(1)


  Test 1: Multi-class Classification (10 classes)
  Loss  : 2.4377 → 0.0021
  Acc   : 0.1031 → 1.0000
  ✓ PASS  Loss decreased over training
  ✓ PASS  Accuracy increased over training
  ✓ PASS  Loss is positive (not NaN/zero)
  ✓ PASS  History has 'loss' key
  ✓ PASS  History has 'val_loss' key
  ✓ PASS  History length matches epochs

  Test 2: Binary Classification (linearly separable)
  Loss  : 0.6243 → 0.0005
  Acc   : 0.6417 → 1.0000
  ✓ PASS  Loss decreased
  ✓ PASS  Final accuracy > 80%
  ✓ PASS  No NaN in loss history

  Test 3: Regression (sine wave)
  MSE   : 0.2916 → 0.0006
  MAE   : 0.4409 → 0.0198
  ✓ PASS  MSE decreased
  ✓ PASS  Final MAE < 0.15
  ✓ PASS  No NaN in loss history

  Test 4: Gradient Sanity Check
  Loss value      : 1.9799
  Graph nodes     : 18
  Parameter nodes : 4
  ✓ PASS  Computation graph reaches model Parameters
  Layer 0  W grad norm: 0.694011  b grad norm: 0.295033
  ✓ PASS  Layer 0 W has non-zero gradient
  ✓ PASS  Layer 0 b has non-zero gradient


In [ ]:
# problem solved